# Find the patient's name: demo

The heuristic lives in [`heuristic.py`](heuristic.py): typed functions, with the input checked by Pydantic.
Its unit tests are in [`tests/`](tests) and run on every push with GitHub Actions. This notebook only shows how to use it.

1. `load_document(path)` reads a JSON file and checks it: a missing field, or a coordinate outside [0, 1], raises an error.
2. `read_words(page)` puts the words back in reading order: line by line, each line from left to right.
3. `find_patient(document)` tries rule 1 (the `Nom:` and `Prénom:` labels), then rule 2 (the first civility that does not follow a doctor title), and returns a `PatientName`, or `None`.

In [1]:
from heuristic import find_patient, load_document, read_words

document = load_document("examples/example3.json")
print(" ".join(read_words(document.pages[0]))[:150], "...")
print(find_patient(document))

Dr William ECLANCHER 16 rue Charles Tellier 75016 Paris IPP : 123456789123 NIR : 1 9308 75000 123 00 Date de la consultation: 13/12/2019 J’ai bien rev ...
first_name='Jean' last_name='DUPONT'


In [2]:
from pathlib import Path

# The 3 challenge examples, the short sample, and the 4 documents where this version fails.
for path in [Path("example.json"), *sorted(Path("examples").glob("**/*.json"))]:
    print(f"{str(path):50} {find_patient(load_document(path))}")

example.json                                       first_name='Jean' last_name='DUPONT'
examples/example1.json                             first_name='Juliette' last_name='MARTIN'
examples/example2.json                             first_name='Victor' last_name='Hugo'
examples/example3.json                             first_name='Jean' last_name='DUPONT'
examples/issues/issue1_letterhead_civility.json    first_name='Sophie' last_name='DURAND'
examples/issues/issue2_hospital_sticker.json       None
examples/issues/issue3_identity_block.json         None
examples/issues/issue4_relative_first.json         first_name='Claire' last_name='PETIT'


In [3]:
from pydantic import ValidationError

from heuristic import Document

# Boxes in pixels instead of [0, 1]: Pydantic refuses the document before the heuristic runs.
try:
    Document.model_validate({"pages": [{"words": [
        {"text": "Monsieur", "bbox": {"x_min": 120, "x_max": 180, "y_min": 40, "y_max": 52}},
    ]}]})
except ValidationError as error:
    print(error)

4 validation errors for Document
pages.0.words.0.bbox.x_min
  Input should be less than or equal to 1 [type=less_than_equal, input_value=120, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
pages.0.words.0.bbox.x_max
  Input should be less than or equal to 1 [type=less_than_equal, input_value=180, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
pages.0.words.0.bbox.y_min
  Input should be less than or equal to 1 [type=less_than_equal, input_value=40, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
pages.0.words.0.bbox.y_max
  Input should be less than or equal to 1 [type=less_than_equal, input_value=52, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/less_than_equal
